In [1]:
pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


# Load datasets

In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

# Fetch dataset
bcwd = fetch_ucirepo(id=17)

# Extract features and target
X_bcwd = bcwd.data.features
y_bcwd = bcwd.data.targets

# Convert target to 0 and 1
label_encoder_bcwd = LabelEncoder()
y_bcwd = label_encoder_bcwd.fit_transform(y_bcwd).ravel() 

# Ensure data is in pandas DataFrame format
if not isinstance(X_bcwd, pd.DataFrame):
    X_bcwd = pd.DataFrame(X_bcwd)
if not isinstance(y_bcwd, pd.Series):
    if isinstance(y_bcwd, pd.DataFrame):
        y_bcwd = y_bcwd.iloc[:, 0]  # Extract the first column if y is a DataFrame
    else:
        y_bcwd = pd.Series(y_bcwd)

# Combine features and target into one DataFrame
bcwd = pd.concat([X_bcwd, y_bcwd.rename("target")], axis=1)





# Fetch dataset
bcwo = fetch_ucirepo(id=15)

# Extract features and target
X_bcwo = bcwo.data.features
y_bcwo = bcwo.data.targets

# Convert target to 0 and 1
label_encoder_bcwo = LabelEncoder()
y_bcwo = label_encoder_bcwo.fit_transform(y_bcwo).ravel() 

# Ensure data is in pandas DataFrame format
if not isinstance(X_bcwo, pd.DataFrame):
    X_bcwo = pd.DataFrame(X_bcwo)
if not isinstance(y_bcwo, pd.Series):
    if isinstance(y_bcwo, pd.DataFrame):
        y_bcwo = y_bcwo.iloc[:, 0]  # Extract the first column if y is a DataFrame
    else:
        y_bcwo = pd.Series(y_bcwo)

# Combine features and target into one DataFrame
bcwo = pd.concat([X_bcwo, y_bcwo.rename("target")], axis=1)




bct= pd.read_csv('/Users/mosesaderounmu/Desktop/bcancerfeat.csv')

bct = bct[bct['target'] != 3]

X_bct = bct.drop(columns=['target'])

y_bct = bct['target']

label_encoder_bct = LabelEncoder()

y_bct = label_encoder_bct.fit_transform(y_bct)

bct['target'] = y_bct



/Users/mosesaderounmu/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/Users/mosesaderounmu/opt/anaconda3/lib/python3.9/site-packages/sklearn/preprocessing/_label.py:116: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/mosesaderounmu/opt/anaconda3/lib/python3.9/site-packages/sklearn/preprocessing/_label.py:116: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


# Weighted SVC

In [25]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.feature_selection import RFE
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import make_scorer, recall_score, confusion_matrix, accuracy_score, f1_score
import warnings

warnings.filterwarnings('ignore')

def ensure_dataframe(X, y):
    """
    Ensures that the features (X) and target (y) are in pandas DataFrame or Series format.

    Parameters:
    - X: Features, can be a numpy array, list, or DataFrame. Will be converted to a DataFrame if not already.
    - y: Target, can be a numpy array, list, Series, or DataFrame. 
         If a DataFrame is provided, the first column is extracted as a Series.

    Returns:
    - X: Features as a pandas DataFrame.
    - y: Target as a pandas Series.
    """
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)
    if not isinstance(y, pd.Series):
        if isinstance(y, pd.DataFrame):
            y = y.iloc[:, 0]  # Extract the first column if y is a DataFrame
        else:
            y = pd.Series(y)
    return X, y


X_bcwd, y_bcwd = ensure_dataframe(X_bcwd, y_bcwd)
X_bcwo, y_bcwo = ensure_dataframe(X_bcwo, y_bcwo)

# Impute missing values for BCWO dataset using KNNImputer
imputer = KNNImputer(n_neighbors=5)
X_bcwo = pd.DataFrame(imputer.fit_transform(X_bcwo), columns=X_bcwo.columns)

# Define custom G-Mean and specificity scores
def specificity_score(y_true, y_pred):
    """
    Calculates the specificity (true negative rate) of the predictions.

    Specificity is the proportion of actual negatives that are correctly identified 
    by the model, which is calculated as the ratio of true negatives (TN) to the sum of 
    true negatives and false positives (TN + FP).

    Parameters:
    -----------
    y_true : array-like
        True binary labels. These should be a 1D array or list of the actual labels.
    
    y_pred : array-like
        Predicted binary labels. These should be a 1D array or list of the predicted labels by the model.

    Returns:
    --------
    specificity : float
        The specificity score, which is a float value between 0 and 1. It is calculated as TN / (TN + FP).
    """
    cm = confusion_matrix(y_true, y_pred)
    tn = cm[0, 0]
    fp = cm[0, 1]
    return tn / (tn + fp)

def geometric_mean_custom_score(y_true, y_pred):
    """
    Computes the geometric mean of sensitivity (recall) and specificity.

    The geometric mean is a useful metric in imbalanced datasets, as it considers both the true positive rate and the true negative rate.
    
    Parameters:
    - y_true: Array of true labels.
    - y_pred: Array of predicted labels.

    Returns:
    - gmean: The geometric mean of sensitivity and specificity.
    """
    sensitivity = recall_score(y_true, y_pred)
    specificity = specificity_score(y_true, y_pred)
    return np.sqrt(sensitivity * specificity)

g_mean_scorer = make_scorer(geometric_mean_score, greater_is_better=True)
specificity = make_scorer(specificity_score)
sensitivity = make_scorer(recall_score)
accuracy = make_scorer(accuracy_score)
f1 = make_scorer(f1_score)

n_features_values = {
    'BCWD': [10, 15, 20, 25, 30],
    'BCWO': [5, 6, 7, 8, 9],
    'BCT': [14, 20, 26, 32, 38]
}

# Define the parameter grid for GridSearchCV
def get_param_grid_for_dataset(dataset_name, include_rfe=True):
    """
    Generates a parameter grid for GridSearchCV based on the dataset name.

    Parameters:
    - dataset_name: The name of the dataset.
    - include_rfe: Whether to include RFE in the pipeline.

    Returns:
    - param_grid: Dictionary of parameters for GridSearchCV.
    """
    param_grid = {
        'svc__kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
        'svc__C': [0.1, 0.5, 1, 5, 7, 10],
        'svc__gamma': ['scale', 0.01, 0.1, 0.3, 0.5, 1.0]
    }
    if include_rfe:
        param_grid['rfe__n_features_to_select'] = n_features_values[dataset_name]
    
    return param_grid

# Function to run GridSearchCV with or without RFE for SVM model
def run_grid_search(X, y, dataset_name, include_rfe=True):
    """
    Runs GridSearchCV for SVM model with optional RFE to find the best hyperparameters.

    Parameters:
    - X: Features as a DataFrame.
    - y: Target as a Series.
    - dataset_name: Name of the dataset, used to retrieve n_features_values.
    - include_rfe: Whether to include RFE in the pipeline.

    Returns:
    - grid_search: The fitted GridSearchCV object with the best parameters and model.
    """
    scaler = StandardScaler()
    svc = SVC(class_weight='balanced', probability=True)

    # Create a pipeline with or without RFE
    if include_rfe:
        rfe = RFE(estimator=svc)
        pipeline = Pipeline([
            ('scaler', scaler),
            ('rfe', rfe),
            ('svc', svc)
        ])
        param_grid = {
            'svc__kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
            'svc__C': [0.1, 0.5, 1, 5, 7, 10],
            'svc__gamma': ['scale', 0.01, 0.1, 0.3, 0.5, 1.0],
            'rfe__n_features_to_select': n_features_values[dataset_name]  # Only include RFE parameter here
        }
    else:
        pipeline = Pipeline([
            ('scaler', scaler),
            ('svc', svc)
        ])
        # Define the parameter grid for SVM only
        param_grid = {
            'svc__kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
            'svc__C': [0.1, 0.5, 1, 5, 7, 10],
            'svc__gamma': ['scale', 0.01, 0.1, 0.3, 0.5, 1.0]
        }

    stratified_kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=101)

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring=g_mean_scorer,
        cv=stratified_kfold,
        n_jobs=2,
        verbose=1
    )

    grid_search.fit(X, y)
    
    return grid_search

datasets = {
    'BCWD': (X_bcwd, y_bcwd),
    'BCWO': (X_bcwo, y_bcwo),
    'BCT': (X_bct, y_bct)
}

results = []

for dataset_name, (X, y) in datasets.items():
    # Run the grid search with RFE
    grid_search_with_rfe = run_grid_search(X, y, dataset_name, include_rfe=True)
    # Run the grid search without RFE
    grid_search_without_rfe = run_grid_search(X, y, dataset_name, include_rfe=False)

    for grid_search, method in [(grid_search_with_rfe, 'With RFE'), (grid_search_without_rfe, 'Without RFE')]:
        # Extract the best model and evaluate using cross-validation
        best_model = grid_search.best_estimator_
        stratified_kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=101)

        cv_results = cross_validate(best_model, X, y, cv=stratified_kfold, scoring={
            'accuracy': accuracy,
            'sensitivity': sensitivity,
            'specificity': specificity,
            'g_mean': g_mean_scorer,
            'f1': f1
        }, return_train_score=False)

        # Determine number of features selected or used
        num_features = (best_model.named_steps['rfe'].n_features_ if method == 'With RFE' else X.shape[1])

        result = {
            'Dataset': dataset_name,
            'Method': method,
            'Accuracy': np.mean(cv_results['test_accuracy']),
            'Sensitivity': np.mean(cv_results['test_sensitivity']),
            'Specificity': np.mean(cv_results['test_specificity']),
            'G-Mean': np.mean(cv_results['test_g_mean']),
            'F1 Score': np.mean(cv_results['test_f1']),
            'Best Params': grid_search.best_params_,
            'Num Features Selected': num_features
        }
        results.append(result)

results_df = pd.DataFrame(results).round(4)
print(results_df)


Fitting 10 folds for each of 720 candidates, totalling 7200 fits


/Users/mosesaderounmu/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/Users/mosesaderounmu/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Fitting 10 folds for each of 144 candidates, totalling 1440 fits
Fitting 10 folds for each of 720 candidates, totalling 7200 fits
Fitting 10 folds for each of 144 candidates, totalling 1440 fits
Fitting 10 folds for each of 720 candidates, totalling 7200 fits
Fitting 10 folds for each of 144 candidates, totalling 1440 fits
  Dataset       Method  Accuracy  Sensitivity  Specificity  G-Mean  F1 Score  \
0    BCWD     With RFE    0.9842       0.9714       0.9917  0.9814    0.9783   
1    BCWD  Without RFE    0.9842       0.9714       0.9917  0.9814    0.9783   
2    BCWO     With RFE    0.9713       0.9833       0.9650  0.9740    0.9596   
3    BCWO  Without RFE    0.9713       0.9833       0.9650  0.9740    0.9596   
4     BCT     With RFE    0.7276       0.6667       0.7462  0.6571    0.4866   
5     BCT  Without RFE    0.7276       0.6667       0.7462  0.6571    0.4866   

                                         Best Params  Num Features Selected  
0  {'rfe__n_features_to_select': 30,

# Weighted ANN

### Please note: the result output is just above the plot confusion matrix key params error message

In [21]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from boruta import BorutaPy
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score, accuracy_score, confusion_matrix, f1_score
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
from hyperopt import hp, tpe, fmin, Trials, STATUS_OK
import warnings

# Close any existing TensorFlow sessions
tf.keras.backend.clear_session()

# Configure TensorFlow to allow GPU memory growth
config = tf.compat.v1.ConfigProto()
config.gpu_options.allow_growth = True
session = tf.compat.v1.InteractiveSession(config=config)

warnings.filterwarnings('ignore')

def ensure_dataframe(X, y):
    """
    Ensures that the features (X) and target (y) are in pandas DataFrame or Series format.

    Parameters:
    - X: Features, can be a numpy array, list, or DataFrame. Will be converted to a DataFrame if not already.
    - y: Target, can be a numpy array, list, Series, or DataFrame. 
         If a DataFrame is provided, the first column is extracted as a Series.

    Returns:
    - X: Features as a pandas DataFrame.
    - y: Target as a pandas Series.
    """
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)
    if not isinstance(y, pd.Series):
        if isinstance(y, pd.DataFrame):
            y = y.iloc[:, 0]  # Extract the first column if y is a DataFrame
        else:
            y = pd.Series(y)
    return X, y

X_bcwd, y_bcwd = ensure_dataframe(X_bcwd, y_bcwd)
X_bcwo, y_bcwo = ensure_dataframe(X_bcwo, y_bcwo)

# Impute missing values for BCWO dataset using KNNImputer
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors=5)
X_bcwo = pd.DataFrame(imputer.fit_transform(X_bcwo), columns=X_bcwo.columns)

def specificity_score(y_true, y_pred):
    """
    Computes the specificity (true negative rate) from true and predicted labels.

    Specificity measures the proportion of actual negatives that are correctly identified.
    
    Parameters:
    - y_true: Array of true labels.
    - y_pred: Array of predicted labels.

    Returns:
    - specificity: The specificity, calculated as TN / (TN + FP).
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

def geometric_mean_custom_score(y_true, y_pred):
    """
    Computes the geometric mean of sensitivity (recall) and specificity.

    The geometric mean is a useful metric in imbalanced datasets, as it considers both the true positive rate and the true negative rate.
    
    Parameters:
    - y_true: Array of true labels.
    - y_pred: Array of predicted labels.

    Returns:
    - gmean: The geometric mean of sensitivity and specificity.
    """
    sensitivity = recall_score(y_true, y_pred)
    specificity = specificity_score(y_true, y_pred)
    return np.sqrt(sensitivity * specificity)

def specificity(y_true, y_pred):
    """
    Computes the specificity (true negative rate) in a TensorFlow/Keras environment.

    This function is compatible with TensorFlow's tensor operations and can be used as a custom metric in model training.
    
    Parameters:
    - y_true: Tensor of true labels.
    - y_pred: Tensor of predicted labels.

    Returns:
    - specificity: Tensor representing the specificity.
    """
    y_true = tf.cast(y_true, tf.float32)
    true_negatives = tf.reduce_sum(tf.round(tf.clip_by_value((1 - y_true) * (1 - y_pred), 0, 1)))
    possible_negatives = tf.reduce_sum(tf.round(tf.clip_by_value(1 - y_true, 0, 1)))
    return true_negatives / (possible_negatives + tf.keras.backend.epsilon())

def gmean(y_true, y_pred):
    """
    Computes the geometric mean of sensitivity and specificity in a TensorFlow/Keras environment.

    This function is designed to work as a custom metric within TensorFlow, making it suitable for evaluating models trained on imbalanced datasets.
    
    Parameters:
    - y_true: Tensor of true labels.
    - y_pred: Tensor of predicted labels.

    Returns:
    - gmean: Tensor representing the geometric mean of sensitivity and specificity.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    true_positives = tf.reduce_sum(tf.round(tf.clip_by_value(y_true * y_pred, 0, 1)))
    possible_positives = tf.reduce_sum(tf.round(tf.clip_by_value(y_true, 0, 1)))
    sensitivity = true_positives / (possible_positives + tf.keras.backend.epsilon())
    
    spec = specificity(y_true, y_pred)
    return tf.sqrt(sensitivity * spec)

def create_ann(hidden_layer_sizes, activation, optimizer, dropout_rate, input_dim, learning_rate, class_weight):
    """
    Creates and compiles a weighted Artificial Neural Network (ANN) model with specified hyperparameters.

    This model is designed to handle imbalanced datasets by applying class weights to the loss function.

    Parameters:
    - hidden_layer_sizes: Tuple of integers representing the number of units in each hidden layer (e.g., (256, 128, 64)).
    - activation: String representing the activation function to use (e.g., 'relu', 'tanh').
    - optimizer: String representing the optimizer to use (e.g., 'adam', 'sgd').
    - dropout_rate: Float representing the dropout rate to apply after each layer to prevent overfitting.
    - input_dim: Integer representing the number of input features.
    - learning_rate: Float representing the learning rate for the optimizer.
    - class_weight: Dictionary specifying the weight for each class (e.g., {0: 1, 1: 3}).

    Returns:
    - model: A compiled Keras Sequential model ready for training.
    """
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Dense(hidden_layer_sizes[0], activation=activation, input_shape=(input_dim,), kernel_regularizer=tf.keras.regularizers.l2(0.001)))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.Dropout(dropout_rate))
    
    for units in hidden_layer_sizes[1:]:
        model.add(tf.keras.layers.Dense(units, activation=activation, kernel_regularizer=tf.keras.regularizers.l2(0.001)))
        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.Dropout(dropout_rate))
    
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    
    optimizer_instance = tf.keras.optimizers.get(optimizer)
    optimizer_instance.learning_rate = learning_rate
    
    model.compile(optimizer=optimizer_instance, 
                  loss='binary_crossentropy', 
                  metrics=['accuracy', tf.keras.metrics.Recall(name='sensitivity'), specificity, gmean])
    
    return model

space = {
    'hidden_layer_sizes': hp.choice('hidden_layer_sizes', [(256, 128, 64, 32), (512, 256, 128), (128, 64, 32), (64, 32)]),
    'activation': hp.choice('activation', ['relu', 'tanh']),
    'optimizer': hp.choice('optimizer', ['adam', 'sgd']),
    'dropout_rate': hp.uniform('dropout_rate', 0.3, 0.7),
    'learning_rate': hp.loguniform('learning_rate', np.log(0.0001), np.log(0.01)),
    'class_weight': hp.choice('class_weight', [
        {0: 1, 1: 2}, 
        {0: 1, 1: 3},  
        {0: 1, 1: 4},  
        {0: 1, 1: 5}   
    ]),
    'use_feature_selection': hp.choice('use_feature_selection', [True, False])  # Added option to use or skip feature selection
}

datasets = {
    'BCWD': (X_bcwd, y_bcwd),
    'BCWO': (X_bcwo, y_bcwo),
    'BCT': (X_bct, y_bct)
}

def boruta_feature_selection(X, y):
    """
    Performs feature selection using the Boruta algorithm with a Random Forest model.

    Boruta is a wrapper algorithm that works on top of any classification algorithm, extending the idea of random forests by evaluating the importance of each feature in predicting the outcome.

    Parameters:
    - X: Features as a pandas DataFrame.
    - y: Target as a pandas Series.

    Returns:
    - X_selected: DataFrame with selected features that are deemed important by the Boruta algorithm.
    """
    model_rf = RandomForestClassifier(n_jobs=-1, max_depth=5, random_state=101)
    feat_selector = BorutaPy(model_rf, n_estimators='auto', verbose=0, random_state=101)
    feat_selector.fit(X.values, y.values)
    
    # Select only the features that passed the test
    X_selected = X.iloc[:, feat_selector.support_]
    return X_selected

def objective(params, X, y):
    """
    Objective function for Hyperopt optimization, including feature selection and weighted ANN training.

    Parameters:
    - params: Dictionary of hyperparameters for the ANN model.
    - X: Features as a pandas DataFrame.
    - y: Target as a pandas Series.

    Returns:
    - Dictionary with:
        - loss: Negative of the G-mean score (used for optimization).
        - status: STATUS_OK (indicates that the trial was successful).
        - metrics: Dictionary containing average accuracy, sensitivity, specificity, G-mean, and F1 scores.
        - num_features_selected: The number of features selected by Boruta.
    """
    hidden_layer_sizes = params['hidden_layer_sizes']
    activation = params['activation']
    optimizer = params['optimizer']
    dropout_rate = params['dropout_rate']
    learning_rate = params['learning_rate']
    class_weight = params['class_weight']
    use_feature_selection = params.get('use_feature_selection', True)  # Default to True

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    if use_feature_selection:
        # Apply Boruta feature selection
        X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
        X_selected = boruta_feature_selection(X_scaled_df, pd.Series(y))
    else:
        # Use all features without selection
        X_selected = pd.DataFrame(X_scaled, columns=X.columns)

    num_features_selected = X_selected.shape[1]  # Calculate the number of selected features

    model = create_ann(hidden_layer_sizes, activation, optimizer, dropout_rate, num_features_selected, learning_rate, class_weight)

    early_stopping = EarlyStopping(monitor='val_gmean', patience=20, restore_best_weights=True, mode='max')

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=101)
    metrics = {'accuracy': [], 'sensitivity': [], 'specificity': [], 'gmean': [], 'f1': []}
    
    for train_idx, test_idx in cv.split(X_selected, y):
        # Convert indices to numpy arrays
        train_idx = np.array(train_idx)
        test_idx = np.array(test_idx)

        # Ensure y is a pandas Series
        if not isinstance(y, pd.Series):
            y = pd.Series(y)

        # Ensure consistency of indices
        X_train, X_test = X_selected.iloc[train_idx].reset_index(drop=True), X_selected.iloc[test_idx].reset_index(drop=True)
        y_train, y_test = y.iloc[train_idx].reset_index(drop=True), y.iloc[test_idx].reset_index(drop=True)

        model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=100, batch_size=32,
                  callbacks=[early_stopping], verbose=0, class_weight=class_weight)

        # Predict and compute metrics
        predictions = (model.predict(X_test) > 0.5).astype(int)
        accuracy = accuracy_score(y_test, predictions)
        sensitivity = recall_score(y_test, predictions)
        spec = specificity_score(y_test, predictions)
        g_mean = geometric_mean_custom_score(y_test, predictions)
        f1 = f1_score(y_test, predictions)

        metrics['accuracy'].append(accuracy)
        metrics['sensitivity'].append(sensitivity)
        metrics['specificity'].append(spec)
        metrics['gmean'].append(g_mean)
        metrics['f1'].append(f1)

    # Average metrics
    avg_metrics = {k: np.mean(v) for k, v in metrics.items()}
    
    return {'loss': -avg_metrics['gmean'], 'status': STATUS_OK, 'metrics': avg_metrics, 'num_features_selected': num_features_selected}

results = []
for dataset_name, (X, y) in datasets.items():
    trials_with_fs = Trials()
    
    # Run the search with feature selection
    best_params_with_fs = fmin(fn=lambda params: objective(params, X, y),
                               space=space,
                               algo=tpe.suggest,
                               max_evals=50,  # Adjust this as needed for more extensive searching
                               trials=trials_with_fs,
                               rstate=np.random.default_rng(101))
    
    # Store results with feature selection
    trial_result_with_fs = trials_with_fs.best_trial['result']['metrics']
    num_features_selected_with_fs = trials_with_fs.best_trial['result']['num_features_selected']
    trial_result_with_fs.update({
        'Dataset': dataset_name,
        'Method': f"{dataset_name} Weighted ANN with Feature Selection",
        'Features Selected': num_features_selected_with_fs
    })
    results.append(trial_result_with_fs)
    
    trials_without_fs = Trials()
    
    # Run the search without feature selection
    best_params_without_fs = fmin(fn=lambda params: objective({**params, 'use_feature_selection': False}, X, y),
                                  space=space,
                                  algo=tpe.suggest,
                                  max_evals=50,  # Adjust this as needed for more extensive searching
                                  trials=trials_without_fs,
                                  rstate=np.random.default_rng(101))
    
    # Store results without feature selection
    trial_result_without_fs = trials_without_fs.best_trial['result']['metrics']
    trial_result_without_fs.update({
        'Dataset': dataset_name,
        'Method': f"{dataset_name} Weighted ANN without Feature Selection",
        'Features Selected': X.shape[1]  # All features used
    })
    results.append(trial_result_without_fs)

results_ann = pd.DataFrame(results)
print(results_ann.round(4))


import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Find the best result with feature selection.
best_with_fs = results_ann[results_ann['Method'].str.contains("with Feature Selection")].iloc[0]

# Rebuild the best model using the best parameters.
best_params = best_with_fs['Best Params']  

# Recreate the model
best_model = create_ann(
    hidden_layer_sizes=best_params['hidden_layer_sizes'],
    activation=best_params['activation'],
    optimizer=best_params['optimizer'],
    dropout_rate=best_params['dropout_rate'],
    input_dim=int(best_with_fs['Features Selected']),  # Number of features selected
    learning_rate=best_params['learning_rate']
)

# Scale the data again
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply Boruta feature selection (assuming the same function as before)
X_selected = boruta_feature_selection(pd.DataFrame(X_scaled, columns=X.columns), y)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=101)

for train_idx, test_idx in cv.split(X_selected.values, y):
    X_train, X_test = X_selected.values[train_idx], X_selected.values[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Train the best model
    best_model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=0)

    # Predict and plot the confusion matrix
    y_pred = (best_model.predict(X_test) > 0.5).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title(f'Confusion Matrix for {best_with_fs["Dataset"]} with Feature Selection')
    plt.show()
    break  



# Close the TensorFlow session
session.close()


1/2 ━━━━━━━━━━━━━━━━━━━━ 1s 2s/step           
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step           
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step           

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step        
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step        
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step         

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step        
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step        
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 222ms/step        

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step         
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step         

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step         
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step         

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step         
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step         

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step         
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step         

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 263ms/step        
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step       
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 264ms/step       

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step        
2/2 ━━━━━

KeyError: 'Best Params'